# Assistência do GPT e tempo até o merge de pull requests

**MCEQ 12576 – Projeto Final (PUC-Campinas)** · Rafael Roveri Pires e Felipe Nonato Leoneli

Análise confirmatória definida no protocolo de 1 página. Estudo **observacional**, sem afirmações causais.

- **H0:** μA = μB · **H1 (unilateral):** μA < μB, em que A = PRs com GPT e B = PRs sem GPT
- **VD:** `time_to_merge_h`, o tempo até o merge em horas · **VI:** `is_gpt` (1 = com GPT, 0 = sem GPT)
- **Teste principal:** Mann-Whitney U unilateral, α = 0,05 · **Efeito:** Cliff's delta + diferença de medianas com IC de 95% (bootstrap)
- **Sensibilidade:** Welch unilateral sobre log(1 + horas), com d de Cohen

Fonte dos dados: Collante et al., *The Impact of Large Language Models (LLMs) on Code Review Process*, arXiv 2508.11034 (v2).
Repositório `acollant/GPT-Assistance-PR`, commit fixado abaixo.

In [ ]:
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ALPHA = 0.05
SEED = 2026
N_BOOT = 10_000

COMMIT = "72df511069ee0255c44460839de55b69c9a5549a"  # versão fixada da base (acesso em 25/11/2026)
URL = f"https://raw.githubusercontent.com/acollant/GPT-Assistance-PR/{COMMIT}/R/RQ1/at_resolution.csv"

BRUTO = Path("dados/brutos/at_resolution.csv")
LIMPO = Path("dados/limpos/prs_limpos.csv")
FIG = Path("figuras/boxplot_tempo_merge.png")
for p in (BRUTO, LIMPO, FIG):
    p.parent.mkdir(parents=True, exist_ok=True)

## 1. Download dos dados brutos
O CSV não é redistribuído neste repositório: ele é baixado da fonte original, no commit fixado. Os dados brutos nunca são editados.

In [ ]:
if not BRUTO.exists():
    urllib.request.urlretrieve(URL, BRUTO)
bruto = pd.read_csv(BRUTO)
print(bruto.shape)
bruto.head()

## 2. Limpeza (regras do protocolo)
Remove apenas: tempo ausente, tempo negativo, `is_gpt` fora de {0, 1} e PRs duplicados (mesmo projeto e número). **Nenhum outlier é removido.**

In [ ]:
df = bruto.copy()
log_limpeza = {"linhas brutas": len(df)}

df = df.dropna(subset=["time_to_merge_h", "is_gpt"])
log_limpeza["após remover tempo/grupo ausente"] = len(df)

df = df[df["time_to_merge_h"] >= 0]
log_limpeza["após remover tempo negativo"] = len(df)

df = df[df["is_gpt"].isin([0, 1])]
log_limpeza["após remover is_gpt inválido"] = len(df)

df = df.drop_duplicates(subset=["project", "pull_number"])
log_limpeza["após remover duplicados"] = len(df)

df["grupo"] = df["is_gpt"].map({1: "A: com GPT", 0: "B: sem GPT"})
df.to_csv(LIMPO, index=False)

for etapa, n in log_limpeza.items():
    print(f"{etapa:35s} {n:6d}")
print("linhas removidas no total:", len(bruto) - len(df))

## 3. Estatística descritiva
VD por grupo e variáveis de controle por grupo (para checar o balanceamento do pareamento feito pelos autores).

In [ ]:
a = df.loc[df["is_gpt"] == 1, "time_to_merge_h"].to_numpy()
b = df.loc[df["is_gpt"] == 0, "time_to_merge_h"].to_numpy()

def descreve(x):
    return pd.Series({"n": len(x), "média": x.mean(), "desvio": x.std(ddof=1),
                      "Q1": np.percentile(x, 25), "mediana": np.median(x),
                      "Q3": np.percentile(x, 75), "mín": x.min(), "máx": x.max()})

desc = pd.DataFrame({"A: com GPT": descreve(a), "B: sem GPT": descreve(b)}).T
print("Tempo até o merge (horas)")
desc.round(2)

In [ ]:
controles = ["PR_size(SLOC)", "no_commits", "no_changed_files", "project_age_month"]
df.groupby("grupo")[controles].median().round(1).rename_axis("mediana por grupo")

## 4. Figura
Boxplot em escala logarítmica, com os pontos visíveis (a escala log é necessária pela assimetria; está indicada no eixo).

In [ ]:
rng_plot = np.random.default_rng(SEED)
fig, ax = plt.subplots(figsize=(5, 4))
dados = [a + 1, b + 1]  # +1 h para permitir PRs com tempo 0 na escala log
ax.boxplot(dados, showfliers=False, widths=0.5, zorder=3, patch_artist=True,
           boxprops=dict(facecolor="none"), medianprops=dict(color="black", linewidth=2))
for i, x in enumerate(dados, start=1):
    ax.scatter(i + rng_plot.uniform(-0.15, 0.15, len(x)), x, s=4, alpha=0.25, zorder=1)
ax.set_yscale("log")
ax.set_xticks([1, 2], [f"Com GPT (n={len(a)})", f"Sem GPT (n={len(b)})"])
ax.set_ylabel("Tempo até o merge + 1 (horas, escala log)")
ax.set_title("Tempo até o merge por grupo")
fig.tight_layout()
fig.savefig(FIG, dpi=300)
plt.show()

## 5. Teste confirmatório: Mann-Whitney U unilateral
H1: tempos do grupo A tendem a ser menores que os do grupo B (`alternative='less'`).

In [ ]:
mw = stats.mannwhitneyu(a, b, alternative="less")
cliff = 2 * mw.statistic / (len(a) * len(b)) - 1  # Cliff's delta = P(A>B) - P(A<B); negativo => A menor

rng = np.random.default_rng(SEED)
boot = (np.median(rng.choice(a, (N_BOOT, len(a))), axis=1)
        - np.median(rng.choice(b, (N_BOOT, len(b))), axis=1))
ic_low, ic_high = np.percentile(boot, [2.5, 97.5])
dif_med = np.median(a) - np.median(b)

print(f"U = {mw.statistic:.1f}, p (unilateral) = {mw.pvalue:.3g}")
print(f"Decisão com α = {ALPHA}: {'rejeita H0' if mw.pvalue < ALPHA else 'não rejeita H0'}")
print(f"Cliff's delta = {cliff:.3f}")
print(f"Diferença de medianas (A − B) = {dif_med:.2f} h, IC 95% bootstrap [{ic_low:.2f}; {ic_high:.2f}] h")

Referência usual para |Cliff's delta| (Romano et al., 2006): < 0,147 desprezível; < 0,33 pequeno; < 0,474 médio; ≥ 0,474 grande.

## 6. Análise de sensibilidade (pré-registrada): Welch sobre log(1 + horas)

In [ ]:
la, lb = np.log1p(a), np.log1p(b)
welch = stats.ttest_ind(la, lb, equal_var=False, alternative="less")
d = (la.mean() - lb.mean()) / np.sqrt((la.var(ddof=1) + lb.var(ddof=1)) / 2)
ic = welch.confidence_interval(confidence_level=0.95)  # unilateral: limite inferior = -inf

print(f"t = {welch.statistic:.3f}, gl = {welch.df:.1f}, p (unilateral) = {welch.pvalue:.3g}")
print(f"d de Cohen (escala log) = {d:.3f}")
print(f"Diferença de médias em log(1+h) = {la.mean() - lb.mean():.3f}; limite superior unilateral 95% = {ic.high:.3f}")
print(f"Razão das médias geométricas de (1+h), A/B = {np.exp(la.mean() - lb.mean()):.2f}")

## 7. Verificações automáticas
Garante que a análise rodou sobre os dados esperados.

In [ ]:
assert set(df["is_gpt"].unique()) == {0, 1}
assert len(a) > 0 and len(b) > 0
assert len(a) + len(b) == len(df)
assert -1 <= cliff <= 1
assert ic_low <= ic_high
print("OK: verificações passaram")

## Análises exploratórias
Qualquer análise além das seções 5 e 6 deve ficar **abaixo desta linha** e ser relatada no artigo como exploratória.